<img src="https://raw.githubusercontent.com/brazil-data-cube/code-gallery/master/img/logo-bdc.png" align="right" width="64"/>

# <span style="color:#336699">Introduction to the SpatioTemporal Asset Catalog (STAC)</span>
<hr style="border:2px solid #0077b9;">

<div style="text-align: left;">
    <a href="https://nbviewer.jupyter.org/github/brazil-data-cube/code-gallery/blob/master/jupyter/Python/stac/stac-introduction.ipynb"><img src="https://raw.githubusercontent.com/jupyter/design/master/logos/Badges/nbviewer_badge.svg" align="center"/></a>
</div>

<br/>

<div style="text-align: center;font-size: 90%;">
    Rennan Marujo<sup><a href="https://orcid.org/0000-0002-0082-9498"><i class="fab fa-lg fa-orcid" style="color: #a6ce39"></i></a></sup>
    <br/><br/>
    Earth Observation and Geoinformatics Division, National Institute for Space Research (INPE)
    <br/>
    Avenida dos Astronautas, 1758, Jardim da Granja, São José dos Campos, SP 12227-010, Brazil
    <br/><br/>
    Contact: <a href="mailto:rennan.marujo@inpe.br">rennan.marujo@inpe.br</a>
    <br/><br/>
    Last Update: Mar 26, 2026
</div>

<br/>

<div style="text-align: justify;  margin-left: 25%; margin-right: 25%;">
<b>Abstract.</b> This Jupyter Notebook gives an overview on how to use the STAC service to discover and access the data products from the <em>National Institute for Space Research (INPE)</em>.
</div>

<img src="https://raw.githubusercontent.com/brazil-data-cube/code-gallery/master/img/stac/stac.png?raw=true" align="right" width="66"/>

# Introduction
<hr style="border:1px solid #0077b9;">

The [**S**patio**T**emporal **A**sset **C**atalog (STAC)](https://stacspec.org/) is a specification created through the colaboration of several organizations intended to increase satellite image search interoperability.

The diagram depicted in the picture contains the most important concepts behind the STAC data model:

<center>
<img src="https://raw.githubusercontent.com/brazil-data-cube/code-gallery/master/img/stac/stac-concept.png" width="480" />
<br/>
<b>Figure 1</b> - STAC model.
</center>

The description of the concepts below are adapted from the [STAC Specification](https://github.com/radiantearth/stac-spec):

- **Item**: a `STAC Item` is the atomic unit of metadata in STAC, providing links to the actual `assets` (including thumbnails) that they represent. It is a `GeoJSON Feature` with additional fields for things like time, links to related entities and mainly to the assets. According to the specification, this is the atomic unit that describes the data to be discovered in a `STAC Catalog` or `Collection`.

- **Asset**: a `spatiotemporal asset` is any file that represents information about the earth captured in a certain space and time.


- **Catalog**: provides a structure to link various `STAC Items` together or even to other `STAC Catalogs` or `Collections`.


- **Collection:** is a specialization of the `Catalog` that allows additional information about a spatio-temporal collection of data.

# STAC Client API
<hr style="border:1px solid #0077b9;">

For running the examples in this Jupyter Notebook you will need to install the [pystac-client](https://pystac-client.readthedocs.io/en/latest/). To install it from PyPI using `pip`, use the following command:

In [ ]:
# !pip install pystac-client

In [ ]:
# !pip install rasterio shapely matplotlib tqdm

In order to access the funcionalities of the client API, you should import the `stac` package, as follows:

In [ ]:
import pystac_client

After that, you can check the installed `stac` package version:

In [ ]:
pystac_client.__version__

Then, create a `STAC` object attached to the Brazil Data Cube' STAC service:

In [ ]:
service = pystac_client.Client.open('https://data.inpe.br/bdc/stac/v1/')

# Listing the Available Data Products
<hr style="border:1px solid #0077b9;">

In the Jupyter environment, the `STAC` object will list the available image and data cube collections from the service:

In [ ]:
for collection in service.get_collections():
    print(collection)

<img src="https://raw.githubusercontent.com/brazil-data-cube/code-gallery/master/img/stac/stac-catalog.png?raw=true" align="right" width="300"/>

# Retrieving the Metadata of a Collection
<hr style="border:1px solid #0077b9;">

The `collection` method returns information about a given image or data cube collection identified by its name. In this example we are retrieving information about the Sentinel-2 Level-2A product `S2_L2A-1`:

In [ ]:
collection = service.get_collection('S2_L2A-1')
collection

<img src="https://raw.githubusercontent.com/brazil-data-cube/code-gallery/master/img/stac/stac-item.png?raw=true" align="right" width="300"/>

# Retrieving Items
<hr style="border:1px solid #0077b9;">

The `get_items` method returns the items of a given collection:

In order to support filtering rules through the specification of a rectangle (`bbox`) or a date and time (`datatime`) criterias, use the `Client.search(**kwargs)`:

In [ ]:
item_search = service.search(bbox=(-61.7960,-9.0374,-61.7033,-8.9390),
                             datetime='2024-08-01/2025-07-31',
                             collections=['S2_L2A-1'])
item_search

The method `.search(**kwargs)` returns a `ItemSearch` representation which has handy methods to identify the matched results. For example, to check the number of items matched, use `.matched()`:

In [ ]:
item_search.matched()

To iterate over the matched result, use `.items()` to traverse the list of items:

In [ ]:
for item in item_search.items():
    print(item)

You can also search using other properties of metadata in the STAC, for instance Cloud cover or tile id. You can even use multiples!

In [ ]:
collection_name = "S2_L2A-1"

item_search = service.search(collections=[collection_name],
                             bbox=(-61.7960,-9.0374,-61.7033,-8.9390),
                             datetime='2024-08-01/2025-07-31',
                             query={
                                 "eo:cloud_cover": {"lt": 10}
                                 }
                            )
print(item_search.matched())

In [ ]:
item_search = service.search(collections=[collection_name],
                             datetime='2024-08-01/2025-07-31',
                             query={
                                 "tileId": {"in": ["22JHT", "22JGT", "22KHU", "22KGU"]}
                                 }
                            )
print(item_search.matched())

In [ ]:
item_search = service.search(collections=[collection_name],
                             datetime='2024-08-01/2025-07-31',
                             query={
                                 "tileId": {"in": ["22JHT", "22JGT", "22KHU", "22KGU"]},
                                 "eo:cloud_cover": {"lt": 10}
                                 }
                            )

print(item_search.matched())

You can also search for data using a geometry.

In [ ]:
import geopandas as gpd

In [ ]:
arquivo_gleba = '/home/jovyan/Aula/Input_data/Vetores/Glebas/glebas_selecionadas_100_maiores.shp'
ref_bacen = 515791395

In [ ]:
glebas = gpd.read_file(arquivo_gleba)
glebas.head(3)

In [ ]:
gleba = glebas[glebas['REF_BACEN'] == ref_bacen]
gleba

In [ ]:
gleba_4326 = gleba.to_crs('EPSG:4326').iloc[0]
gleba_4326

In [ ]:
geom = gleba_4326.geometry

bounds = geom.bounds

print(f"Geometria: {geom}")
print()
print(f"Tipo da geometria: {geom.geom_type}")
print()
print(f"Bounds: {bounds}")

In [ ]:
import folium
from shapely.geometry import box

bounds_polygon = box(*bounds)

centro_lat = (bounds[1] + bounds[3]) / 2
centro_lon = (bounds[0] + bounds[2]) / 2

m = folium.Map(location=[centro_lat, centro_lon], zoom_start=15)

folium.GeoJson(
    geom,
    name="Gleba",
    style_function=lambda x: {
        'fillColor': 'green',
        'color': 'darkgreen',
        'weight': 2,
        'fillOpacity': 0.5
    }
).add_to(m)

folium.GeoJson(
    bounds_polygon.__geo_interface__,
    name="Bounds",
    style_function=lambda x: {
        'fillColor': 'none',
        'color': 'blue',
        'weight': 3,
        'fillOpacity': 0
    }
).add_to(m)

m.fit_bounds([[bounds[1], bounds[0]], [bounds[3], bounds[2]]])
m

In [ ]:
item_search = service.search(intersects=geom,
                             datetime='2024-08-01/2025-07-31',
                             collections=['S2_L2A-1'])
print(item_search.matched())

In [ ]:
item_list = list(item_search.items())

In [ ]:
for item in item_list:
    print(item)

<img src="https://raw.githubusercontent.com/brazil-data-cube/code-gallery/master/img/stac/stac-asset.png?raw=true" align="right" width="300"/>

# Assets
<hr style="border:1px solid #0077b9;">

The assets with the links to the images, thumbnails or specific metadata files, can be accessed through the property `assets` (from a given item):

In [ ]:
item_n = 0
print(item_list[item_n])
item = item_list[item_n]
assets = item.assets

Then, from the assets it is possible to traverse or access individual elements:

In [ ]:
for k in assets.keys():
    print(k)

The metadata related to the Sentinel-2/MSI blue band is available under the dictionary key `B02`:

In [ ]:
blue_asset = assets['B02']
blue_asset

To iterate in the item's assets, use the following pattern:

In [ ]:
for asset in assets.values():
    print(asset)

# Using RasterIO and NumPy
<hr style="border:1px solid #0077b9;">

The `rasterio` library can be used to read image files from the Brazil Data Cube' service on-the-fly and then to create `NumPy` arrays. The `read` method of an `Item` can be used to perform the reading and array creation:

In [ ]:
import rasterio

In [ ]:
with rasterio.open(assets['B04'].href) as red_ds:
    red = red_ds.read(1)

<div style="text-align: justify;  margin-left: 15%; margin-right: 15%; border-style: solid; border-color: #0077b9; border-width: 1px; padding: 5px;">
    <b>Note:</b> If there are errors because of your pyproj version, you can run the code below as specified in <a  href="https://rasterio.readthedocs.io/en/latest/faq.html#why-can-t-rasterio-find-proj-db-rasterio-from-pypi-versions-1-2-0" target="_blank">rasterio documentation</a> and try again:

       import os
       del os.environ['PROJ_LIB']
</div>

In [ ]:
red

In [ ]:
with rasterio.open(assets['B03'].href) as green_ds:
    green = green_ds.read(1)
with rasterio.open(assets['B02'].href) as blue_ds:
    blue = blue_ds.read(1)

# Using Matplotlib to Visualize Images
<hr style="border:1px solid #0077b9;">

The `Matplotlib` cab be used to plot the arrays read in the last section:

In [ ]:
%matplotlib inline
from matplotlib import pyplot as plt

In [ ]:
fig, (ax1, ax2, ax3) = plt.subplots(1,3, figsize=(12, 4))
ax1.imshow(red, cmap='gray')
ax2.imshow(green, cmap='gray')
ax3.imshow(blue, cmap='gray')

Using `Numpy` we can stack the previous arrays and use `Matplotlib` to plot a color image, but first we need to normalize their values:

In [ ]:
import numpy

def normalize(array, brightness_factor=1):
    """Normalizes numpy arrays with optional brightness adjustment"""
    array_min, array_max = array.min(), array.max()
    normalized = (array - array_min) / (array_max - array_min)
    return numpy.clip(normalized * brightness_factor, 0, 1)

In [ ]:
brightness_factor = 1
rgb = numpy.dstack((
    normalize(red, brightness_factor), 
    normalize(green, brightness_factor), 
    normalize(blue, brightness_factor))
)
plt.imshow(rgb)

# Using Window
<hr style="border:1px solid #0077b9;">

The next cell code import the `Window` class from the `rasterio` library in order to retrieve a subset of an image and then create an array:

We have prepared a basic function `read()`to read raster windows as [`numpy.ma.masked_array`](https://numpy.org/doc/stable/reference/maskedarray.generic.html).

We can specify a subset of the image file (window or chunck) to be read. Let's read a range that starts on pixel (0, 0) with 500 x 500 and column 0 to column 500, for the spectral bands `red`, `green` and `blue`:

In [ ]:
from rasterio.windows import Window

In [ ]:
def read(uri: str, window: Window, masked: bool = True):
    """Read raster window as numpy.ma.masked_array."""
    with rasterio.open(uri) as ds:
        return ds.read(1, window=window, masked=masked)

In [ ]:
blue

In [ ]:
red = read(assets['B04'].href, window=Window(0, 0, 500, 500)) # Window(col_off, row_off, width, height)

In [ ]:
green = read(assets['B03'].href, window=Window(0, 0, 500, 500))

In [ ]:
blue = read(assets['B02'].href, window=Window(0, 0, 500, 500))

You can also load using Coordinates:

In [ ]:
from rasterio.windows import from_bounds

In [ ]:
from pyproj import Transformer

def transform_bounds(bounds, from_crs='EPSG:4326', to_crs=None):
    transformer = Transformer.from_crs(from_crs, to_crs, always_xy=True)
    
    minx, miny = transformer.transform(bounds[0], bounds[1])
    maxx, maxy = transformer.transform(bounds[2], bounds[3])
    
    return (minx, miny, maxx, maxy)

In [ ]:
with rasterio.open(assets['B02'].href) as src:
    reprojected_bounds = transform_bounds(bounds, 'EPSG:4326', src.crs)
    window = from_bounds(
        reprojected_bounds[0],  # left
        reprojected_bounds[1],  # bottom  
        reprojected_bounds[2],  # right
        reprojected_bounds[3],  # top
        src.transform
    )
    window = window.round_offsets().round_lengths()
    
    blue = read(assets['B02'].href, window=window, masked=True)
    green = read(assets['B03'].href, window=window, masked=True)
    red = read(assets['B04'].href, window=window, masked=True)
print(red.shape)

In [ ]:
brightness_factor = 1
rgb = numpy.dstack((
    normalize(red, brightness_factor), 
    normalize(green, brightness_factor), 
    normalize(blue, brightness_factor))
)
plt.imshow(rgb)

# Using TMS
<hr style="border:1px solid #0077b9;">

In [ ]:
from shapely.geometry import mapping

camada_gleba = folium.GeoJson(
    data=mapping(gleba_4326.geometry),  # Converte shapely para GeoJSON
    name="Polígono da Gleba",
    style_function=lambda x: {
        'color': 'SteelBlue',      # Cor da borda
        'opacity': 1,              # Opacidade da borda
        'fillOpacity': 0.1,        # Opacidade do preenchimento
        'weight': 5,               # Espessura da borda
        'fillColor': 'SteelBlue'   # Cor do preenchimento (mesmo da borda)
    },
    highlight_function=lambda x: {
        'color': 'IndianRed',
        'opacity': 1,
        'fillOpacity': 0.1,
        'weight': 5,
        'fillColor': 'IndianRed'
    }
)

In [ ]:
camada_cena = folium.TileLayer(
    tiles=f"https://data.inpe.br/bdc/tms/tiles/WebMercatorQuad/{{z}}/{{x}}/{{y}}?url={assets['TCI'].href}",
    attr='INPE',
    name="Cena Satélite",
    overlay=True
)

In [ ]:
m = folium.Map(
    location=[centro_lat, centro_lon],
    zoom_start=15
)

camada_gleba.add_to(m)
camada_cena.add_to(m)

folium.TileLayer('OpenStreetMap', name='Mapa Base', control=True).add_to(m)
folium.LayerControl(collapsed=False).add_to(m)
# m.fit_bounds([[bounds[1], bounds[0]], [bounds[3], bounds[2]]])
m

## Using WPM

In [ ]:
collection_name = "CB4A-WPM-PCA-FUSED-1"

item_search = service.search(collections=[collection_name],
                             intersects=geom,
                             datetime='2024-08-01/2025-07-31',
                             query={
                                 "eo:cloud_cover": {"lt": 10}
                                 }
                            )
print(item_search.matched())

item_list = list(item_search.items())

item_n = 1
item = item_list[item_n]
assets = item.assets
assets

In [ ]:
m = folium.Map(
    location=[centro_lat, centro_lon],
    zoom_start=15
)

camada_gleba.add_to(m)
camada_cena = folium.TileLayer(
    tiles=f"https://data.inpe.br/bdc/tms/tiles/WebMercatorQuad/{{z}}/{{x}}/{{y}}?url={assets['tci'].href}",
    attr='INPE',
    name="Cena Satélite",
    overlay=True
)
camada_cena.add_to(m)

folium.TileLayer('OpenStreetMap', name='Mapa Base', control=True).add_to(m)
folium.LayerControl(collapsed=False).add_to(m)
m

# Retrieving Image Files
<hr style="border:1px solid #0077b9;">

The file related to an asset can be retrieved through the `download` method. The cell code below shows ho to download the image file associated to the asset into a folder named `img`:

In [ ]:
# import os
# from urllib.parse import urlparse

# import requests
# from pystac import Asset
# from tqdm import tqdm

# def download(asset: Asset, directory: str = None, chunk_size: int = 1024 * 16, **request_options) -> str:
#     """Smart download STAC Item asset.

#     This method uses a checksum validation and a progress bar to monitor download status.
#     """
#     if directory is None:
#         directory = ''

#     response = requests.get(asset.href, stream=True, **request_options)
#     output_file = os.path.join(directory, urlparse(asset.href)[2].split('/')[-1])
#     os.makedirs(directory, exist_ok=True)
#     total_bytes = int(response.headers.get('content-length', 0))
#     with tqdm.wrapattr(open(output_file, 'wb'), 'write', miniters=1, total=total_bytes, desc=os.path.basename(output_file)) as fout:
#         for chunk in response.iter_content(chunk_size=chunk_size):
#             fout.write(chunk)

In [ ]:
# download(assets['BAND15'], 'img')

In order to download all files related to an item, iterate over assets and download each one as following:

In [ ]:
# for asset in assets.values():
#     download(asset, 'images')

# References
<hr style="border:1px solid #0077b9;">

- [Spatio Temporal Asset Catalog Specification](https://stacspec.org/)


- [Python Client Library for STAC Service](https://pystac-client.readthedocs.io/en/latest/)

# See also the following Jupyter Notebooks
<hr style="border:1px solid #0077b9;">

* [Image processing on images obtained through STAC](./stac-image-processing.ipynb)